# Notebook 03 — Layer 2A Experiments
Train Isolation Forest and Shallow Autoencoder. Compare on FPR / TPR / AUC. Export winner to ONNX.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
import numpy as np
import mlflow
from pathlib import Path

from feature_engineering.normalizer import FeatureNormalizer
from layer2a.candidates.isolation_forest import IsolationForestModel
from layer2a.candidates.autoencoder_shallow import ShallowAutoencoderModel
from layer2a.evaluate import evaluate_candidate, pick_best, threshold_sweep
from evaluation.compare_models import compare_l2a, pick_best_l2a
from evaluation.benchmark import benchmark_isolation_forest, benchmark_autoencoder

SPLITS  = Path("../data/splits")
EXPORTS = Path("../exported_models")
EXPORTS.mkdir(exist_ok=True)

## 1. Load splits

In [ ]:
X_n_train = np.load(SPLITS / "l2a_normal_train.npy")
X_n_val   = np.load(SPLITS / "l2a_normal_val.npy")
X_test    = np.load(SPLITS / "l2a_test_X.npy")
y_test    = np.load(SPLITS / "l2a_test_y.npy")

norm = FeatureNormalizer.load(str(EXPORTS / "scaler_l2a.pkl"))
X_train = norm.transform(X_n_train)
X_val   = norm.transform(X_n_val)
X_te    = norm.transform(X_test)

print(f"Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_te.shape}")
print(f"Test attack rate: {y_test.mean():.2%}")

## 2. Candidate 1 — Isolation Forest

In [ ]:
mlflow.set_experiment("layer2a")
iforest = IsolationForestModel()
iforest.train(X_train)
iforest.tune_threshold(X_val, y_val=None, target_fpr=0.05)
res_iforest = evaluate_candidate(iforest, X_te, y_test, name="isolation_forest")
print(res_iforest)

## 3. Candidate 2 — Shallow Autoencoder

In [ ]:
ae = ShallowAutoencoderModel()
ae.train(X_train, X_val)
res_ae = evaluate_candidate(ae, X_te, y_test, name="shallow_autoencoder")
print(res_ae)

## 4. Compare candidates

In [ ]:
all_results = [res_iforest, res_ae]
df_compare = compare_l2a(all_results)
df_compare

## 5. Pick winner and export ONNX

In [ ]:
winner_name, winner_model = pick_best_l2a(
    all_results,
    models={"isolation_forest": iforest, "shallow_autoencoder": ae},
    max_fpr=0.05,
)
print(f"Winner: {winner_name}")

onnx_path = str(EXPORTS / "layer2a_best.onnx")
winner_model.export_onnx(onnx_path)

## 6. Latency benchmark

In [ ]:
if winner_name == "isolation_forest":
    lat = benchmark_isolation_forest(onnx_path)
else:
    lat = benchmark_autoencoder(onnx_path)

print(f"Mean: {lat['mean_ms']}ms  P99: {lat['p99_ms']}ms  Pass: {lat['pass']}")

## 7. ROC curve

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve

fig, ax = plt.subplots(figsize=(7, 5))
for name, model, result in [
    ("Isolation Forest", iforest, res_iforest),
    ("Shallow Autoencoder", ae, res_ae),
]:
    scores   = model.anomaly_scores(X_te)
    fpr, tpr, _ = roc_curve(y_test, scores)
    ax.plot(fpr, tpr, label=f"{name} (AUC={result['auc']})")

ax.plot([0,1],[0,1],"--", color="gray")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("Layer 2A — ROC Curves")
ax.legend()
plt.tight_layout()
plt.savefig("../data/processed/03_l2a_roc.png", dpi=120)
plt.show()
print(f"Winner exported to: {onnx_path}")